# Time Value of Money: Discounting, Annuities, NPV, IRR

A franc today is worth more than a franc next year, because today's franc can earn interest in the meantime. Discounting turns future
cash flows into today's value; adding them up gives the net present value (NPV) of a project, a loan or a bond. The internal rate of
return (IRR) runs the question backwards: which discount rate makes the NPV zero? There is no formula for it in general, so it is found
numerically, with Newton's method. This notebook derives the annuity formulas from a geometric series, computes NPV and IRR, and shows
when the IRR misleads.

## 0. Math link

- [Newton's method](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/05_newtons_method/00_newtons_method.ipynb)
  and [an example](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/05_newtons_method/01_newtons_method_an_example.ipynb):
  the IRR is a root of the NPV function, found by following tangent lines.
- [Derivatives of polynomials](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/01_derivatives_of_polynomials.ipynb):
  NPV is a polynomial in the discount factor $v = 1/(1+r)$, and Newton's method needs its derivative.
- Compounding conventions from [00/02](./02_continuous_compounding_as_a_limit.ipynb). Rates here are effective annual rates (annual
  compounding) unless stated otherwise.

## 1. Introduction

A company can spend CHF 1 million on a machine that returns CHF 300,000 a year for four years. Is it worth it? The answer depends on
what else the million could earn. Discounting the four payments at that alternative rate and subtracting the cost gives the NPV; a
positive NPV means the machine beats the alternative. The IRR is the rate at which the machine merely breaks even.

After this notebook you can:

- discount single cash flows and streams of cash flows;
- derive and use the formulas for annuities, perpetuities and growing perpetuities;
- build a loan amortization schedule;
- compute NPV and find the IRR with Newton's method and with bisection;
- recognize cash-flow patterns with no IRR or several IRRs, and NPV–IRR ranking conflicts.

## 2. Theory

### Present value

At an annual rate $r$, a cash flow $C_t$ received in $t$ years is worth today

$$
PV = \frac{C_t}{(1 + r)^t} = C_t\, v^t, \qquad v = \frac{1}{1 + r} \text{ (the one-year discount factor)} .
$$

### Net present value

For cash flows $C_0, C_1, \dots, C_n$ (negative for payments, positive for receipts),

$$
\text{NPV}(r) = \sum_{t=0}^{n} \frac{C_t}{(1 + r)^t} = \sum_{t=0}^{n} C_t v^t .
$$

As a function of $v$ it is a polynomial of degree $n$.

### Annuities from a geometric series

A level payment $C$ at the end of each of $n$ years is worth $PV = C(v + v^2 + \dots + v^n)$. The geometric sum is
$v\,\frac{1 - v^n}{1 - v}$, and $1 - v = r v$, so

$$
PV_{\text{annuity}} = C\,\frac{1 - (1 + r)^{-n}}{r} = C\, a_{n|r} .
$$

Letting $n \to \infty$ gives the **perpetuity** $PV = C/r$. A payment that grows at rate $g < r$ each year, starting at $C$, is a
geometric series with ratio $(1 + g)/(1 + r)$:

$$
PV_{\text{growing perpetuity}} = \frac{C}{r - g} .
$$

Solving the annuity formula for $C$ gives the level payment that repays a loan $L$ in $n$ years: $C = L / a_{n|r}$.

### Internal rate of return

The IRR is a rate $r^*$ with $\text{NPV}(r^*) = 0$. For a project that costs money first and pays later ($C_0 < 0$, $C_t \ge 0$ after),
NPV falls steadily in $r$ and there is exactly one IRR. Accept the project if $r^*$ exceeds the cost of capital, which is the same as
$\text{NPV} > 0$ at that cost of capital.

### Newton's method

There is no closed form for $r^*$ when $n > 4$ (and an unpleasant one for $n \le 4$). Newton's method replaces NPV near a guess $r_k$ by
its tangent line and moves to the tangent's zero:

$$
r_{k+1} = r_k - \frac{\text{NPV}(r_k)}{\text{NPV}'(r_k)}, \qquad \text{NPV}'(r) = -\sum_{t=1}^{n} \frac{t\, C_t}{(1 + r)^{t+1}} .
$$

Near a simple root the error roughly squares at each step (quadratic convergence), so a handful of iterations gives machine precision.
The derivative is, up to sign and a factor $1 + r$, the cash-flow-weighted average time: the **duration** of module 01.

### When the IRR fails

- **Several sign changes.** By Descartes' rule of signs, the polynomial $\sum_t C_t v^t$ has at most as many positive roots as there are
  sign changes in $C_0, C_1, \dots$. A project with cash flows $-, +, -$ (e.g. a mine with clean-up costs) can have two IRRs, or none.
- **Ranking.** Two projects can be ranked differently by IRR and NPV when their sizes or timings differ; the IRR ignores scale and assumes
  reinvestment at the IRR itself. NPV at the cost of capital is the correct criterion.

## 3. Worked Examples

### Example 1: the machine

Cost CHF 1,000,000, four payments of CHF 300,000, cost of capital 6%. The annuity factor is
$a_{4|0.06} = (1 - 1.06^{-4})/0.06 = 3.4651$, so $PV = 1{,}039{,}532$ and $\text{NPV} = 39{,}532$: accept.

### Example 2: Newton by hand

Start at $r_0 = 6\%$. $\text{NPV}(0.06) = 39{,}532$ and $\text{NPV}'(0.06) = -300{,}000 \times (1/1.06^2 + 2/1.06^3 + 3/1.06^4 + 4/1.06^5) = -2{,}380{,}365$. Next guess
$r_1 = 0.06 + 39{,}532/2{,}380{,}365 = 0.0766$. One more step gives $0.07714$, the IRR to five digits.

### Example 3: a mortgage-style loan

A CHF 400,000 loan at 2.5% repaid by equal annual payments over 20 years: $C = 400{,}000 / a_{20|0.025} = 25{,}659$ a year. In year 1 the
interest is CHF 10,000 and the repayment CHF 15,659; by the last year almost the whole payment is repayment.

### Example 4: two IRRs

Cash flows $-100$, $+230$, $-132$. With $x = 1 + r$: $-100x^2 + 230x - 132 = 0$, so $x = 1.1$ or $1.2$: IRRs of 10% and 20%. The NPV is
positive only between them, so the project is worth doing at a cost of capital of 15% but not at 5% or 25%. "The IRR" is undefined.

### Example 5: a perpetuity

A property yields a net rent of CHF 60,000 a year, growing 1% a year. At a discount rate of 4% the growing-perpetuity value is
$60{,}000/(0.04 - 0.01) = 2{,}000{,}000$. At 3.5% it is 2,400,000: a half-point change in the rate moves the value by 20%.

## 4. Finance Application

### Where it matters

- **Capital budgeting.** Firms accept projects with positive NPV at the cost of capital; IRR is the summary most managers ask for.
- **Loans and mortgages.** Annuity payments, amortization schedules and the effective rate of a loan with fees (an IRR) all come from
  these formulas.
- **Bond pricing.** A bond is an annuity of coupons plus a final repayment; its yield to maturity is an IRR (module 01).
- **Valuation.** Discounted cash-flow models end with a growing perpetuity (the "terminal value"), which often carries most of the value.
- **Private equity and real estate.** Fund performance is reported as an IRR of the cash flows between investors and the fund.

### The usual misreadings

- "The project with the higher IRR is better." Not if it is smaller or shorter; compare NPVs.
- "IRR is the return the investor earns." Only if interim cash flows are reinvested at the IRR, which is rarely possible.
- "The terminal value is a detail." With $g$ close to $r$, $C/(r - g)$ explodes; small changes in either input dominate the valuation.
- "Every project has one IRR." Cash flows with several sign changes can have several or none.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

### NPV, its derivative, and Newton's method

In [ ]:
def npv(rate, cash_flows):
    t = np.arange(len(cash_flows))
    return np.sum(np.asarray(cash_flows) / (1 + rate) ** t)

def npv_prime(rate, cash_flows):
    t = np.arange(len(cash_flows))
    return np.sum(-t * np.asarray(cash_flows) / (1 + rate) ** (t + 1))

def irr_newton(cash_flows, guess=0.05, tol=1e-12, max_iter=50, verbose=False):
    rate = guess
    for k in range(max_iter):
        step = npv(rate, cash_flows) / npv_prime(rate, cash_flows)
        rate -= step
        if verbose:
            print(f"  iteration {k + 1}: rate = {rate:.12f}, NPV = {npv(rate, cash_flows):,.6f}")
        if abs(step) < tol:
            return rate
    raise RuntimeError("Newton's method did not converge")

machine = [-1_000_000] + [300_000] * 4      # CHF, annual
print(f"NPV at 6%: {npv(0.06, machine):,.0f} CHF")
print(f"NPV'(6%): {npv_prime(0.06, machine):,.0f} CHF per unit of rate")
irr = irr_newton(machine, guess=0.06, verbose=True)
print(f"IRR: {irr:.6%}")

The distance to the IRR shrinks from 0.017 (the start) to $5 \times 10^{-4}$, $5 \times 10^{-7}$ and below $10^{-12}$: the number of correct digits roughly doubles each iteration, the
signature of quadratic convergence.

### Bisection as a check

Bisection needs only a bracket $[a, b]$ with NPV of opposite signs. It halves the bracket each step: slower (one binary digit per
iteration) but it cannot diverge.

In [ ]:
def irr_bisection(cash_flows, low=-0.99, high=1.0, tol=1e-12):
    f_low = npv(low, cash_flows)
    assert f_low * npv(high, cash_flows) < 0, "no sign change in the bracket"
    n_iter = 0
    while high - low > tol:
        mid = (low + high) / 2
        if npv(mid, cash_flows) * f_low > 0:
            low, f_low = mid, npv(mid, cash_flows)
        else:
            high = mid
        n_iter += 1
    return (low + high) / 2, n_iter

rate_b, n_b = irr_bisection(machine, 0.0, 0.5)
print(f"bisection: {rate_b:.10%} after {n_b} iterations")

# NPV is a polynomial in x = 1 + r: multiply by x^n
roots = np.roots(machine)                     # coefficients of C_0 x^n + C_1 x^(n-1) + ...
real_roots = roots[np.abs(roots.imag) < 1e-10].real
print("polynomial roots in 1 + r:", np.round(real_roots, 6), "-> IRR", np.round(real_roots - 1, 6))

Three methods give the same IRR. Bisection needs about 40 iterations for the precision Newton reaches in 4. Treating NPV as a polynomial
and asking `np.roots` for all its roots also shows that this project has only one real positive root.

### Annuities and an amortization schedule

In [ ]:
def annuity_factor(rate, n):
    return (1 - (1 + rate) ** -n) / rate

loan, rate, years = 400_000, 0.025, 20       # CHF, annual rate, years
payment = loan / annuity_factor(rate, years)
balance = loan
rows = []
for year in range(1, years + 1):
    interest = balance * rate
    repayment = payment - interest
    balance -= repayment
    rows.append({"year": year, "payment": payment, "interest": interest, "repayment": repayment, "balance": balance})
schedule = pd.DataFrame(rows).set_index("year")
print(f"annual payment: {payment:,.2f} CHF, total interest: {schedule['interest'].sum():,.2f} CHF")
schedule.iloc[[0, 1, 9, 18, 19]]

The balance reaches zero after 20 payments, which confirms the payment formula. Over the life of the loan the borrower pays about
CHF 113,000 in interest, 28% of the amount borrowed.

### Two IRRs, and a project with none

In [ ]:
two_irrs = [-100, 230, -132]
no_irr = [-100, 230, -133]

for name, cf in [("two IRRs", two_irrs), ("no IRR", no_irr)]:
    roots = np.roots(cf)
    print(f"{name}: roots of the polynomial in 1 + r: {np.round(roots, 4)}")

print("Newton from 5%:", round(irr_newton(two_irrs, 0.05), 6), " Newton from 25%:", round(irr_newton(two_irrs, 0.25), 6))

For the first project Newton's method finds 10% or 20% depending on where it starts; neither is "the" IRR. For the second the
polynomial's roots are complex: the NPV is negative at every rate and no IRR exists, although the cash flows differ from the first project by CHF 1 in the last year.

### NPV and IRR disagree

In [ ]:
project_a = [-1000, 1200]               # CHF, one year
project_b = [-1000, 0, 0, 1500]         # CHF, three years
for name, cf in [("A", project_a), ("B", project_b)]:
    print(f"project {name}: IRR {irr_newton(cf):.2%}, NPV at 5% {npv(0.05, cf):,.1f} CHF, NPV at 15% {npv(0.15, cf):,.1f} CHF")

crossover = irr_newton([b - a for a, b in zip(project_a + [0, 0], project_b)], guess=0.1)
print(f"NPV profiles cross at {crossover:.2%}")

Project A has the higher IRR (20% vs 14.5%), but at a 5% cost of capital project B adds more than twice as much value. Below the
crossover rate of about 12% B is better; above it A is. The IRR ranking is right only above the crossover.

## 6. Visualizations

In [ ]:
rates = np.linspace(-0.05, 0.25, 300)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rates * 100, [npv(x, machine) / 1000 for x in rates], label="NPV(r)")
ax.axhline(0, color="k", lw=0.8)

# Newton's tangent steps from a deliberately poor guess
r_k = 0.22
for k in range(3):
    f, fp = npv(r_k, machine), npv_prime(r_k, machine)
    r_next = r_k - f / fp
    xs = np.array([r_k, r_next])
    ax.plot(xs * 100, (f + fp * (xs - r_k)) / 1000, "r--", lw=1)
    ax.plot(r_k * 100, f / 1000, "ro")
    ax.annotate(f"r{k}", (r_k * 100, f / 1000), textcoords="offset points", xytext=(5, 5))
    r_k = r_next
ax.plot(irr * 100, 0, "k*", ms=12, label=f"IRR = {irr:.2%}")
ax.set_title("NPV of the machine and Newton's tangent steps from 22%")
ax.set_xlabel("discount rate (% per year)")
ax.set_ylabel("NPV (CHF thousand)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The NPV curve is decreasing and convex. From 22% the first tangent (dashed) overshoots to about 4%, left of the root. From there
every tangent lies below the convex curve, so each step stays left of the IRR and climbs towards it; the third guess is already on top
of the star.

In [ ]:
rates = np.linspace(0.0, 0.30, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(rates * 100, [npv(x, project_a) for x in rates], label="A: 1 year")
axes[0].plot(rates * 100, [npv(x, project_b) for x in rates], label="B: 3 years")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].axvline(crossover * 100, color="grey", ls=":", label=f"crossover {crossover:.1%}")
axes[0].set_title("NPV profiles: IRR and NPV rankings")
axes[0].set_xlabel("discount rate (% per year)")
axes[0].set_ylabel("NPV (CHF)")
axes[0].legend()

axes[1].plot(rates * 100, [npv(x, two_irrs) for x in rates], label="-100, +230, -132")
axes[1].plot(rates * 100, [npv(x, no_irr) for x in rates], label="-100, +230, -133")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Two IRRs, and none")
axes[1].set_xlabel("discount rate (% per year)")
axes[1].set_ylabel("NPV (CHF)")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Left: the longer project B is more sensitive to the discount rate, so its NPV profile is steeper; the two cross at about 12%. Right:
the NPV of a project with a final payment is hump-shaped; it can cross zero twice (two IRRs) or never.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(schedule.index, schedule["interest"], label="interest")
ax.bar(schedule.index, schedule["repayment"], bottom=schedule["interest"], label="repayment")
ax2 = ax.twinx()
ax2.plot(schedule.index, schedule["balance"] / 1000, "k-o", ms=3, label="balance after payment")
ax.set_title("Amortization of CHF 400,000 at 2.5% over 20 years")
ax.set_xlabel("year")
ax.set_ylabel("payment split (CHF)")
ax2.set_ylabel("outstanding balance (CHF thousand)")
ax.set_ylim(0, 33_000)
ax2.set_ylim(0, 520)
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.show()

Every payment is the same height, but its composition shifts from mostly interest to almost all repayment, and the balance falls slowly
at first and quickly at the end.

In [ ]:
r_disc = 0.04
growths = np.linspace(0.0, 0.035, 100)
plt.figure(figsize=(8, 5))
plt.plot(growths * 100, 60_000 / (r_disc - growths) / 1e6)
plt.title("Growing perpetuity: CHF 60,000 a year at a 4% discount rate")
plt.xlabel("growth rate g (% per year)")
plt.ylabel("present value (CHF million)")
plt.grid(True, alpha=0.3)
plt.show()

The value $C/(r - g)$ rises from CHF 1.5 million at no growth to CHF 12 million at 3.5% growth. When $g$ approaches $r$ the formula
explodes, which is why terminal values in valuation models are so sensitive to the growth assumption.

## 7. Exercises

### Basic

1. What is the present value of CHF 10,000 received in 8 years at 3%? At 3% continuously compounded?
2. A savings plan pays in CHF 6,000 at the end of each year for 30 years at 4%. What is the value after 30 years? (Hint: the future value
   of an annuity is $C\,\frac{(1 + r)^n - 1}{r}$.)
3. A project has an IRR of 18% and the firm's cost of capital is 8%. A manager says "the project earns 18% a year." **Interpret** this
   claim. When is it accurate, and what does the NPV add?

### Intermediate

4. A loan of CHF 20,000 is repaid in 36 monthly payments of CHF 620. Find the monthly IRR with Newton's method and convert it into an
   effective annual rate.
5. Count the sign changes and find all IRRs of the cash flows $-100, +360, -431, +171.6$ (CHF thousand) by computing the roots of the polynomial.
   Plot the NPV profile.

### Advanced

6. Show that for $C_0 < 0$ and $C_t \ge 0$ (not all zero) for $t \ge 1$, $\text{NPV}(r)$ is strictly decreasing and convex on $r > -1$, so the
   IRR is unique if it exists. Why does convexity guarantee that Newton's method started to the left of the root, where NPV is positive,
   never overshoots?
7. Derive the growing-annuity formula $PV = \frac{C}{r - g}\Big[1 - \big(\tfrac{1 + g}{1 + r}\big)^n\Big]$ and check it numerically for
   $C = 100$, $r = 5\%$, $g = 2\%$, $n = 10$.

## 8. Solutions / Checks

In [ ]:
# 1.
print(f"1. annual {10_000 / 1.03**8:,.2f} CHF, continuous {10_000 * np.exp(-0.03 * 8):,.2f} CHF")

# 2.
fv = 6_000 * ((1.04) ** 30 - 1) / 0.04
print(f"2. future value {fv:,.2f} CHF (paid in: {6_000 * 30:,} CHF)")

# 4.
loan_cf = [20_000] + [-620] * 36
monthly = irr_newton(loan_cf, guess=0.005)
print(f"4. monthly IRR {monthly:.5%}, effective annual {(1 + monthly) ** 12 - 1:.3%}")

# 5.
cf5 = [-100, 360, -431, 171.6]
signs = np.sign(cf5)
roots5 = np.roots(cf5)
print("5. sign changes:", int(np.sum(signs[1:] != signs[:-1])), " IRRs:", np.round(np.sort(roots5.real) - 1, 4),
      " max imaginary part:", np.abs(roots5.imag).max())

# 7.
C, r_, g_, n_ = 100, 0.05, 0.02, 10
direct = sum(C * (1 + g_) ** (t - 1) / (1 + r_) ** t for t in range(1, n_ + 1))
formula = C / (r_ - g_) * (1 - ((1 + g_) / (1 + r_)) ** n_)
print(f"7. direct sum {direct:.6f}, formula {formula:.6f}")

In [ ]:
# 5. NPV profile
rates = np.linspace(0.0, 0.4, 400)
plt.figure(figsize=(8, 4.5))
plt.plot(rates * 100, [npv(x, cf5) for x in rates])
plt.axhline(0, color="k", lw=0.8)
plt.title("NPV of -100, +360, -431, +171.6")
plt.xlabel("discount rate (% per year)")
plt.ylabel("NPV (CHF thousand)")
plt.grid(True, alpha=0.3)
plt.show()

**3.** The IRR is the break-even discount rate of the cash flows; it is the yearly return only if every interim cash flow could be
reinvested at 18%. Comparing it with the 8% cost of capital says the project is worth doing, but not by how much. The NPV at 8% measures the
value added in francs, which is what the shareholders get.

**5.** Three sign changes allow up to three IRRs; this polynomial has them, at 10%, 20% and 30%. Between consecutive roots the NPV
changes sign, so the decision depends on which interval the cost of capital falls in.

**6.** $\text{NPV}'(r) = -\sum_t t C_t (1+r)^{-t-1} < 0$ and $\text{NPV}''(r) = \sum_t t(t+1) C_t (1+r)^{-t-2} > 0$, because each term with $C_t > 0$
has the stated sign and $C_0$ does not depend on $r$. A strictly decreasing function crosses zero at most once. For a convex decreasing
function, the tangent at a point left of the root lies below the curve, so it reaches zero before the curve does: each Newton step stays
left of the root and moves towards it monotonically.

**7.** The sum is a geometric series with first term $C/(1+r)$ and ratio $q = (1+g)/(1+r)$, so $PV = \frac{C}{1+r}\cdot\frac{1 - q^n}{1 - q}$,
and $(1 + r)(1 - q) = r - g$. The two printed numbers agree.

## 9. Conclusion

- Present value discounts each cash flow by $(1 + r)^{-t}$; NPV adds them and is a polynomial in $v = 1/(1 + r)$.
- Annuity, perpetuity and growing-perpetuity formulas are geometric series; loan payments are $L / a_{n|r}$.
- The IRR is a root of NPV. Newton's method finds it in a few steps (quadratic convergence); bisection is slower but safe.
- Several sign changes can mean several IRRs or none; NPV and IRR can rank projects differently. Decide with NPV at the cost of capital.
- The NPV derivative is a cash-flow-weighted time, the duration of the next module.

Next: [00/04 Annualizing return and volatility](./04_annualizing_return_and_volatility.ipynb).